In [1]:
from models.decision_tree import train_decision_tree, evaluate_decision_tree
from models.svc import train_svc, evaluate_svc
from models.mlp import train_mlp, evaluate_mlp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import pickle
import copy

In [2]:
historical_data = pd.read_csv("data/historical_data.csv")
current_data = pd.read_csv("data/current_data.csv")

In [3]:
X_train_hist, X_test_hist, y_train_hist, y_test_hist = train_test_split(
    historical_data.drop(["EMERGENCY_ENCOUNTERS", "HIGH_EMERGENCY_RISK", "PATIENT"], axis=1),
    historical_data["HIGH_EMERGENCY_RISK"],
    test_size=0.2,
    random_state=42
)

X_train_curr, X_test_curr, y_train_curr, y_test_curr = train_test_split(
    current_data.drop(["EMERGENCY_ENCOUNTERS", "HIGH_EMERGENCY_RISK", "PATIENT"], axis=1),
    current_data["HIGH_EMERGENCY_RISK"],
    test_size=0.2,
    random_state=42
)

# Support Vector Classifier

In [4]:
svc = train_svc(X_train_hist, y_train_hist, save_path='saved_models/svc_historical.pkl')
matrix, report = evaluate_svc(svc, X_test_hist, y_test_hist)
print("SVC Historical Data Evaluation:")
print(matrix)
print(report)

SVC Historical Data Evaluation:
[[167   5]
 [ 43  13]]
{'0': {'precision': 0.7952380952380952, 'recall': 0.9709302325581395, 'f1-score': 0.8743455497382199, 'support': 172.0}, '1': {'precision': 0.7222222222222222, 'recall': 0.23214285714285715, 'f1-score': 0.35135135135135137, 'support': 56.0}, 'accuracy': 0.7894736842105263, 'macro avg': {'precision': 0.7587301587301587, 'recall': 0.6015365448504983, 'f1-score': 0.6128484505447857, 'support': 228.0}, 'weighted avg': {'precision': 0.7773043720412143, 'recall': 0.7894736842105263, 'f1-score': 0.745890834344954, 'support': 228.0}}


In [5]:
# Grid search
param_grid = {
    'C': [0.1, 1],
    'kernel': ['rbf', 'linear'],
}
svc = SVC(random_state=42)
grid_search = GridSearchCV(estimator=svc, param_grid=param_grid, cv=2, verbose=3)
grid_search.fit(X_train_hist, y_train_hist)
print("\nGrid search completed.\n")
print("Best parameters found: ", grid_search.best_params_)
best_svc = grid_search.best_estimator_
matrix, report = evaluate_svc(best_svc, X_test_hist, y_test_hist)
print("SVC Historical Data Evaluation After Grid Search:")
print(matrix)
print(report)

pickle.dump(best_svc, open('saved_models/best_svc_historical.pkl', 'wb'))

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV 1/2] END .................C=0.1, kernel=rbf;, score=0.747 total time=   0.0s
[CV 2/2] END .................C=0.1, kernel=rbf;, score=0.742 total time=   0.0s
[CV 1/2] END ..............C=0.1, kernel=linear;, score=0.780 total time= 1.9min
[CV 2/2] END ..............C=0.1, kernel=linear;, score=0.764 total time= 3.0min
[CV 1/2] END ...................C=1, kernel=rbf;, score=0.774 total time=   0.0s
[CV 2/2] END ...................C=1, kernel=rbf;, score=0.769 total time=   0.0s
[CV 1/2] END ................C=1, kernel=linear;, score=0.787 total time= 2.2min
[CV 2/2] END ................C=1, kernel=linear;, score=0.764 total time= 2.7min

Grid search completed.

Best parameters found:  {'C': 1, 'kernel': 'linear'}
SVC Historical Data Evaluation After Grid Search:
[[163   9]
 [ 35  21]]
{'0': {'precision': 0.8232323232323232, 'recall': 0.9476744186046512, 'f1-score': 0.8810810810810811, 'support': 172.0}, '1': {'precision': 0.

In [6]:
matrix, report = evaluate_svc(best_svc, X_test_curr, y_test_curr)
print("SVC Current Data Evaluation:")
print(matrix)
print(report)

SVC Current Data Evaluation:
[[170   9]
 [ 35  14]]
{'0': {'precision': 0.8292682926829268, 'recall': 0.9497206703910615, 'f1-score': 0.8854166666666666, 'support': 179.0}, '1': {'precision': 0.6086956521739131, 'recall': 0.2857142857142857, 'f1-score': 0.3888888888888889, 'support': 49.0}, 'accuracy': 0.8070175438596491, 'macro avg': {'precision': 0.7189819724284199, 'recall': 0.6177174780526735, 'f1-score': 0.6371527777777778, 'support': 228.0}, 'weighted avg': {'precision': 0.7818645234507265, 'recall': 0.8070175438596491, 'f1-score': 0.7787067495126704, 'support': 228.0}}


In [7]:
# Finetuning on current data
best_svc_curr = copy.deepcopy(best_svc)
best_svc_curr.fit(X_train_curr, y_train_curr)
matrix, report = evaluate_svc(best_svc_curr, X_test_curr, y_test_curr)
print("Finetuned SVC Current Data Evaluation:")
print(matrix)
print(report)

pickle.dump(best_svc_curr, open('saved_models/finetuned_svc_current.pkl', 'wb'))

Finetuned SVC Current Data Evaluation:
[[169  10]
 [ 33  16]]
{'0': {'precision': 0.8366336633663366, 'recall': 0.9441340782122905, 'f1-score': 0.8871391076115486, 'support': 179.0}, '1': {'precision': 0.6153846153846154, 'recall': 0.32653061224489793, 'f1-score': 0.4266666666666667, 'support': 49.0}, 'accuracy': 0.8114035087719298, 'macro avg': {'precision': 0.726009139375476, 'recall': 0.6353323452285942, 'f1-score': 0.6569028871391076, 'support': 228.0}, 'weighted avg': {'precision': 0.789084525861493, 'recall': 0.8114035087719298, 'f1-score': 0.7881779251277801, 'support': 228.0}}


# Decision Tree

In [8]:
decision_tree = train_decision_tree(X_train_hist, y_train_hist, save_path='saved_models/decision_tree_historical.pkl')
matrix, report = evaluate_decision_tree(decision_tree, X_test_hist, y_test_hist)
print("Decision Tree Historical Data Evaluation:")
print(matrix)
print(report)

Decision Tree Historical Data Evaluation:
[[153  19]
 [ 36  20]]
{'0': {'precision': 0.8095238095238095, 'recall': 0.8895348837209303, 'f1-score': 0.8476454293628809, 'support': 172.0}, '1': {'precision': 0.5128205128205128, 'recall': 0.35714285714285715, 'f1-score': 0.42105263157894735, 'support': 56.0}, 'accuracy': 0.7587719298245614, 'macro avg': {'precision': 0.6611721611721612, 'recall': 0.6233388704318937, 'f1-score': 0.6343490304709141, 'support': 228.0}, 'weighted avg': {'precision': 0.7366493155966841, 'recall': 0.7587719298245614, 'f1-score': 0.7428682509598094, 'support': 228.0}}


In [9]:
# Grid search
param_grid = {
    'max_depth': [4, 6],
    'min_samples_split': [2, 4],
}
decision_tree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(estimator=decision_tree, param_grid=param_grid, cv=2, verbose=3)
grid_search.fit(X_train_hist, y_train_hist)
print("\nGrid search completed.\n")
print("Best parameters found: ", grid_search.best_params_)
best_decision_tree = grid_search.best_estimator_
matrix, report = evaluate_decision_tree(best_decision_tree, X_test_hist, y_test_hist)
print("Decision Tree Historical Data Evaluation After Grid Search:")
print(matrix)
print(report)

pickle.dump(best_decision_tree, open('saved_models/best_decision_tree_historical.pkl', 'wb'))

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV 1/2] END ..max_depth=4, min_samples_split=2;, score=0.767 total time=   0.0s
[CV 2/2] END ..max_depth=4, min_samples_split=2;, score=0.773 total time=   0.0s
[CV 1/2] END ..max_depth=4, min_samples_split=4;, score=0.767 total time=   0.0s
[CV 2/2] END ..max_depth=4, min_samples_split=4;, score=0.773 total time=   0.0s
[CV 1/2] END ..max_depth=6, min_samples_split=2;, score=0.760 total time=   0.0s
[CV 2/2] END ..max_depth=6, min_samples_split=2;, score=0.756 total time=   0.0s
[CV 1/2] END ..max_depth=6, min_samples_split=4;, score=0.760 total time=   0.0s
[CV 2/2] END ..max_depth=6, min_samples_split=4;, score=0.756 total time=   0.0s

Grid search completed.

Best parameters found:  {'max_depth': 4, 'min_samples_split': 2}
Decision Tree Historical Data Evaluation After Grid Search:
[[157  15]
 [ 36  20]]
{'0': {'precision': 0.8134715025906736, 'recall': 0.9127906976744186, 'f1-score': 0.8602739726027397, 'support': 172.0},

In [10]:
matrix, report = evaluate_decision_tree(best_decision_tree, X_test_curr, y_test_curr)
print("Decision Tree Current Data Evaluation:")
print(matrix)
print(report)

Decision Tree Current Data Evaluation:
[[167  12]
 [ 21  28]]
{'0': {'precision': 0.8882978723404256, 'recall': 0.9329608938547486, 'f1-score': 0.9100817438692098, 'support': 179.0}, '1': {'precision': 0.7, 'recall': 0.5714285714285714, 'f1-score': 0.6292134831460674, 'support': 49.0}, 'accuracy': 0.8552631578947368, 'macro avg': {'precision': 0.7941489361702128, 'recall': 0.75219473264166, 'f1-score': 0.7696476135076387, 'support': 228.0}, 'weighted avg': {'precision': 0.8478303471444569, 'recall': 0.8552631578947368, 'f1-score': 0.8497197053804644, 'support': 228.0}}


In [11]:
# Finetuning on current data
best_decision_tree_curr = copy.deepcopy(best_decision_tree)
best_decision_tree_curr.fit(X_train_curr, y_train_curr)
matrix, report = evaluate_decision_tree(best_decision_tree_curr, X_test_curr, y_test_curr)
print("Finetuned Decision Tree Current Data Evaluation:")
print(matrix)
print(report)

pickle.dump(best_decision_tree_curr, open('saved_models/finetuned_decision_tree_current.pkl', 'wb'))

Finetuned Decision Tree Current Data Evaluation:
[[175   4]
 [ 23  26]]
{'0': {'precision': 0.8838383838383839, 'recall': 0.9776536312849162, 'f1-score': 0.9283819628647215, 'support': 179.0}, '1': {'precision': 0.8666666666666667, 'recall': 0.5306122448979592, 'f1-score': 0.6582278481012658, 'support': 49.0}, 'accuracy': 0.881578947368421, 'macro avg': {'precision': 0.8752525252525253, 'recall': 0.7541329380914377, 'f1-score': 0.7933049054829937, 'support': 228.0}, 'weighted avg': {'precision': 0.8801479709374446, 'recall': 0.881578947368421, 'f1-score': 0.8703225259199437, 'support': 228.0}}


# MLP

In [12]:
mlp = train_mlp(X_train_hist, y_train_hist, save_path='saved_models/mlp_historical.pkl')
matrix, report = evaluate_mlp(mlp, X_test_hist, y_test_hist)
print("MLP Historical Data Evaluation:")
print(matrix)
print(report)

MLP Historical Data Evaluation:
[[117  55]
 [ 32  24]]
{'0': {'precision': 0.785234899328859, 'recall': 0.6802325581395349, 'f1-score': 0.7289719626168224, 'support': 172.0}, '1': {'precision': 0.3037974683544304, 'recall': 0.42857142857142855, 'f1-score': 0.35555555555555557, 'support': 56.0}, 'accuracy': 0.618421052631579, 'macro avg': {'precision': 0.5445161838416447, 'recall': 0.5544019933554817, 'f1-score': 0.542263759086189, 'support': 228.0}, 'weighted avg': {'precision': 0.6669871092649643, 'recall': 0.618421052631579, 'f1-score': 0.6372556521105464, 'support': 228.0}}


In [13]:
# Grid search
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)]
}
mlp = MLPClassifier(random_state=42)
grid_search = GridSearchCV(estimator=mlp, param_grid=param_grid, cv=2, verbose=3)
grid_search.fit(X_train_hist, y_train_hist)
print("\nGrid search completed.\n")
print("Best parameters found: ", grid_search.best_params_)
best_mlp = grid_search.best_estimator_
matrix, report = evaluate_mlp(best_mlp, X_test_hist, y_test_hist)
print("MLP Historical Data Evaluation After Grid Search:")
print(matrix)
print(report)

pickle.dump(best_mlp, open('saved_models/best_mlp_historical.pkl', 'wb'))

Fitting 2 folds for each of 3 candidates, totalling 6 fits
[CV 1/2] END ..........hidden_layer_sizes=(50,);, score=0.662 total time=   0.0s
[CV 2/2] END ..........hidden_layer_sizes=(50,);, score=0.678 total time=   0.0s
[CV 1/2] END .........hidden_layer_sizes=(100,);, score=0.393 total time=   0.0s
[CV 2/2] END .........hidden_layer_sizes=(100,);, score=0.458 total time=   0.0s
[CV 1/2] END .......hidden_layer_sizes=(50, 50);, score=0.664 total time=   0.0s
[CV 2/2] END .......hidden_layer_sizes=(50, 50);, score=0.652 total time=   0.0s

Grid search completed.

Best parameters found:  {'hidden_layer_sizes': (50,)}
MLP Historical Data Evaluation After Grid Search:
[[125  47]
 [ 40  16]]
{'0': {'precision': 0.7575757575757576, 'recall': 0.7267441860465116, 'f1-score': 0.7418397626112759, 'support': 172.0}, '1': {'precision': 0.25396825396825395, 'recall': 0.2857142857142857, 'f1-score': 0.2689075630252101, 'support': 56.0}, 'accuracy': 0.618421052631579, 'macro avg': {'precision': 0.50

In [14]:
matrix, report = evaluate_mlp(best_mlp, X_test_curr, y_test_curr)
print("MLP Current Data Evaluation:")
print(matrix)
print(report)

MLP Current Data Evaluation:
[[119  60]
 [ 31  18]]
{'0': {'precision': 0.7933333333333333, 'recall': 0.664804469273743, 'f1-score': 0.723404255319149, 'support': 179.0}, '1': {'precision': 0.23076923076923078, 'recall': 0.3673469387755102, 'f1-score': 0.28346456692913385, 'support': 49.0}, 'accuracy': 0.6008771929824561, 'macro avg': {'precision': 0.512051282051282, 'recall': 0.5160757040246267, 'f1-score': 0.5034344111241414, 'support': 228.0}, 'weighted avg': {'precision': 0.6724313990103464, 'recall': 0.6008771929824561, 'f1-score': 0.6288558135160317, 'support': 228.0}}


In [15]:
# Finetuning on current data
best_mlp_curr = copy.deepcopy(best_mlp)
best_mlp_curr.fit(X_train_curr, y_train_curr)
matrix, report = evaluate_mlp(best_mlp_curr, X_test_curr, y_test_curr)
print("Finetuned MLP Current Data Evaluation:")
print(matrix)
print(report)

pickle.dump(best_mlp_curr, open('saved_models/finetuned_mlp_current.pkl', 'wb'))

Finetuned MLP Current Data Evaluation:
[[125  54]
 [ 24  25]]
{'0': {'precision': 0.8389261744966443, 'recall': 0.6983240223463687, 'f1-score': 0.7621951219512195, 'support': 179.0}, '1': {'precision': 0.31645569620253167, 'recall': 0.5102040816326531, 'f1-score': 0.390625, 'support': 49.0}, 'accuracy': 0.6578947368421053, 'macro avg': {'precision': 0.577690935349588, 'recall': 0.6042640519895108, 'f1-score': 0.5764100609756098, 'support': 228.0}, 'weighted avg': {'precision': 0.7266408524071201, 'recall': 0.6578947368421053, 'f1-score': 0.682340139602054, 'support': 228.0}}
